# Preparación del Dataset: Tratamiento y Limpieza Inicial

Pipeline de limpieza del padrón oficial FIDE (~1.9M registros) para generar dos datasets curados.

## Índice

1. [Carga de Datos Brutos](#1-carga-de-datos-brutos)
2. [Tratamiento de Valores Nulos](#2-tratamiento-de-valores-nulos)
3. [Tipado y Optimización de Memoria](#3-tipado-y-optimización-de-memoria)
4. [Filtrado de Registros Válidos](#4-filtrado-de-registros-válidos)
5. [Control de Duplicados](#5-control-de-duplicados)
6. [Análisis de Flags de Actividad](#6-análisis-de-flags-de-actividad)
7. [Estadísticos Descriptivos Iniciales](#7-estadísticos-descriptivos-iniciales)
8. [Depuración de Año de Nacimiento](#8-depuración-de-año-de-nacimiento)
9. [Ingeniería de Variables: Edad y Categorías](#9-ingeniería-de-variables-edad-y-categorías)
10. [Segmentación de Jugadores Activos](#10-segmentación-de-jugadores-activos)
11. [Exportación de Datasets Procesados](#11-exportación-de-datasets-procesados)


In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt

## 1. Carga de Datos Brutos

Se carga el archivo `fide_players_bruto.parquet` generado por el pipeline de ingesta streaming.
Contiene ~1.9M de registros del padrón mundial FIDE sin filtros.

In [42]:

df = pd.read_parquet('../data/interim/fide_players_bruto.parquet')
print(df.info())
df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 1915099 entries, 0 to 1915098
Data columns (total 12 columns):
 #   Column        Dtype  
---  ------        -----  
 0   fideid        int64  
 1   name          str    
 2   country       str    
 3   sex           str    
 4   title         str    
 5   w_title       str    
 6   o_title       str    
 7   rating        int64  
 8   rapid_rating  int64  
 9   blitz_rating  int64  
 10  birthday      float64
 11  flag          str    
dtypes: float64(1), int64(4), str(7)
memory usage: 216.3 MB
None


,fideid,name,country,sex,title,w_title,o_title,rating,rapid_rating,blitz_rating,birthday,flag
0,10292519,"A A M Imtiaz, Chowdhury",BAN,M,NaN,NaN,NaN,0,0,0,1975.0,NaN
1,10688862,"A Abdel Maabod, Hoda",EGY,F,NaN,NaN,NaN,0,0,0,2009.0,w
2,577017641,A Adhisiva,IND,M,NaN,NaN,NaN,0,0,0,2015.0,NaN
3,577038320,A Akilesh Kumar,IND,M,NaN,NaN,NaN,0,0,0,2015.0,NaN
4,33496722,A Aman,IND,M,NaN,NaN,NaN,0,0,0,1996.0,NaN
5,577037219,A Anuj,IND,M,NaN,NaN,NaN,0,0,0,2015.0,NaN
6,537001345,A Arbhin Vanniarajan,IND,M,NaN,NaN,NaN,1450,1500,1431,2018.0,NaN
7,35853913,"A Aziz, Mohd Azizi Jamil",MAS,M,NaN,NaN,NaN,0,0,0,1981.0,NaN
8,35893303,"A Aziz, Mohd Khalis",MAS,M,NaN,NaN,NaN,0,0,0,1991.0,NaN
9,10224084,"A B M Hasibuzzaman, Tapan",BAN,M,NaN,NaN,NaN,0,0,0,1977.0,NaN


## 2. Tratamiento de Valores Nulos

- **Títulos** (`title`, `w_title`, `o_title`): Los valores `NaN` se imputan como `'nt'` (no title).
- **Flag de actividad** (`flag`): Los valores `NaN` se imputan como `'a'` (activo).

In [43]:
columnas_titulos = ['title','w_title',"o_title"]

for col in columnas_titulos:
    df[col] = df[col].fillna('nt')

df['flag'] = df['flag'].fillna('a')

## 3. Tipado y Optimización de Memoria

Conversión de tipos para optimizar el uso de memoria:
- Variables numéricas (`rating`, `rapid_rating`, `blitz_rating`, `birthday`) → `Int64`
- Variables categóricas (`sex`, `country`, `flag`, títulos) → `category`
- Identificadores (`fideid`, `name`) → `string`

In [44]:
variables_numericas = [
    'rating',
    'rapid_rating',
    'blitz_rating', 
    'birthday'
]

variables_categoricas = [
    'sex',
    'country',
    'flag',
    'w_title',
    'title',
    'o_title'
]   

for var in variables_numericas:
  df[var] = pd.to_numeric(df[var], errors="coerce").astype("Int64")

for col in variables_categoricas:
  df[col] = df[col].astype("category")

df['fideid'] = df['fideid'].astype('string')
df['name'] = df['name'].astype('string')

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1915099 entries, 0 to 1915098
Data columns (total 12 columns):
 #   Column        Dtype   
---  ------        -----   
 0   fideid        string  
 1   name          string  
 2   country       category
 3   sex           category
 4   title         category
 5   w_title       category
 6   o_title       category
 7   rating        Int64   
 8   rapid_rating  Int64   
 9   blitz_rating  Int64   
 10  birthday      Int64   
 11  flag          category
dtypes: Int64(4), category(6), string(2)
memory usage: 156.0 MB


## 4. Filtrado de Registros Válidos

Criterios de filtrado:
- Retener solo jugadores con **al menos un rating > 0** (clásico, rápido o blitz).
- Reemplazar los ratings con valor 0 por `NaN` (indican ausencia de dato, no rating real).
- Eliminar registros sin año de nacimiento (`birthday`).

> Esto descarta ~1.1M de registros con Elo cero (jugadores escolares, árbitros, registros administrativos).

In [45]:
df_limpio = df[
    (df['rating'] > 0) | 
    (df['rapid_rating'] > 0) | 
    (df['blitz_rating'] > 0)
]

df_limpio = df_limpio.replace(0, np.nan)

df_limpio = df_limpio.dropna(subset=['birthday'])


df_limpio.info()


<class 'pandas.DataFrame'>
Index: 776189 entries, 6 to 1915093
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   fideid        776189 non-null  string  
 1   name          776189 non-null  string  
 2   country       776189 non-null  category
 3   sex           776189 non-null  category
 4   title         776189 non-null  category
 5   w_title       776189 non-null  category
 6   o_title       776189 non-null  category
 7   rating        558368 non-null  Int64   
 8   rapid_rating  469675 non-null  Int64   
 9   blitz_rating  311496 non-null  Int64   
 10  birthday      776189 non-null  Int64   
 11  flag          776189 non-null  category
dtypes: Int64(4), category(6), string(2)
memory usage: 69.1 MB


Re-imputación de títulos tras el filtrado para asegurar consistencia.

In [46]:
columnas_titulos = ['title','w_title',"o_title"]

for col in columnas_titulos:
    df_limpio[col] = df_limpio[col].fillna('nt')

## 5. Control de Duplicados

Verificación de unicidad por `fideid`. Se espera que cada jugador tenga un ID FIDE único.

In [47]:
duplicados_id = df_limpio.duplicated(subset=['fideid']).sum()
print(f"Cantidad de registros con FIDE ID repetido: {duplicados_id}")

if duplicados_id > 0:
    filas_repetidas = df_limpio[df_limpio.duplicated(subset=['fideid'], keep=False)]
    display(filas_repetidas)


Cantidad de registros con FIDE ID repetido: 1


,fideid,name,country,sex,title,w_title,o_title,rating,rapid_rating,blitz_rating,birthday,flag
1646141,5019168,Sridharan Ramanathan,IND,M,nt,nt,nt,1841,1702,1782,1965,i
1646142,5019168,Sridharan Ramanathan,IND,M,nt,nt,nt,1841,1702,1782,1965,i


Verificación visual del dataset tras limpieza y deduplicación.

In [48]:
df_limpio.head()

,fideid,name,country,sex,title,w_title,o_title,rating,rapid_rating,blitz_rating,birthday,flag
6,537001345,A Arbhin Vanniarajan,IND,M,nt,nt,nt,1450,1500,1431,2018,a
10,10245154,"A B M Jobair, Hossain",BAN,M,nt,nt,nt,1715,1738,1748,1998,a
15,558074015,A Bala Phani Prabhanjan,IND,M,nt,nt,nt,1585,1553,1524,2012,a
17,12457558,A Bu Ba Co,VIE,M,nt,nt,nt,<NA>,1638,<NA>,2002,a
18,25121731,A C J John,IND,M,nt,nt,nt,1438,<NA>,<NA>,1987,i


## 6. Análisis de Flags de Actividad

Los flags de la FIDE indican el estado del jugador:
- `a` → Activo
- `i` → Inactivo
- `w` → Mujer activa
- `wi` → Mujer inactiva

Se crea la variable booleana `es_activo` derivada de estos flags.

In [49]:
df_flag = df_limpio['flag'].value_counts().reset_index()
df_flag

,flag,count
0,a,395057
1,i,300294
2,w,42199
3,wi,38639


In [50]:
df_limpio['es_activo'] = ~df_limpio['flag'].isin(['i', 'wi'])

df_limpio.info()

<class 'pandas.DataFrame'>
Index: 776189 entries, 6 to 1915093
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   fideid        776189 non-null  string  
 1   name          776189 non-null  string  
 2   country       776189 non-null  category
 3   sex           776189 non-null  category
 4   title         776189 non-null  category
 5   w_title       776189 non-null  category
 6   o_title       776189 non-null  category
 7   rating        558368 non-null  Int64   
 8   rapid_rating  469675 non-null  Int64   
 9   blitz_rating  311496 non-null  Int64   
 10  birthday      776189 non-null  Int64   
 11  flag          776189 non-null  category
 12  es_activo     776189 non-null  bool    
dtypes: Int64(4), bool(1), category(6), string(2)
memory usage: 69.9 MB


## 7. Estadísticos Descriptivos Iniciales

Resumen estadístico del dataset filtrado para validar rangos y detectar anomalías.

In [51]:
df_limpio.describe()

,rating,rapid_rating,blitz_rating,birthday
count,558368.0,469675.0,311496.0,776189.0
mean,1736.543629,1676.65316,1720.147344,1990.935525
std,222.05441,203.955249,214.890101,20.410477
min,1400.0,1400.0,1400.0,15.0
25%,1556.0,1515.0,1549.0,1977.0
50%,1707.0,1635.0,1688.0,1997.0
75%,1881.0,1798.0,1855.0,2008.0
max,2823.0,2803.0,2860.0,2022.0


## 8. Depuración de Año de Nacimiento

Eliminación de registros con `birthday < 1920` como valores ilógicos o errores de registro.

In [52]:
errores_nacimiento = df_limpio[df_limpio['birthday'] < 1920]
df_limpio = df_limpio[df_limpio['birthday'] >= 1920].copy()
print(f"Registros con año de nacimiento ilógico: {len(errores_nacimiento)}")

Registros con año de nacimiento ilógico: 1


## 9. Ingeniería de Variables: Edad y Categorías

### 9.1. Cálculo de Edad
Variable derivada: `edad = año_actual - birthday`.

In [53]:
fecha = dt.datetime.now()
df_limpio['edad'] = fecha.year - df_limpio['birthday'] 

Estadísticos descriptivos tras incorporar la variable `edad`.

In [54]:
df_limpio.describe()

,rating,rapid_rating,blitz_rating,birthday,edad
count,558367.0,469675.0,311496.0,776188.0,776188.0
mean,1736.544092,1676.65316,1720.147344,1990.93807,35.06193
std,222.054339,203.955249,214.890101,20.286892,20.286892
min,1400.0,1400.0,1400.0,1920.0,4.0
25%,1556.0,1515.0,1549.0,1977.0,18.0
50%,1707.0,1635.0,1688.0,1997.0,29.0
75%,1881.0,1798.0,1855.0,2008.0,49.0
max,2823.0,2803.0,2860.0,2022.0,106.0


### 9.2. Inspección de Jugadores Centenarios

Revisión de jugadores nacidos en 1920 (los más antiguos del dataset tras el filtro).

In [55]:
jugador_centenario = df_limpio[df_limpio['birthday'] == 1920]
display(jugador_centenario)

,fideid,name,country,sex,title,w_title,o_title,rating,rapid_rating,blitz_rating,birthday,flag,es_activo,edad
307910,105864,"Carames, Luis",ARG,M,nt,nt,nt,2040,<NA>,<NA>,1920,i,False,106
365419,24515850,"Codina Espinasa, Joan",ESP,M,nt,nt,nt,1695,<NA>,<NA>,1920,i,False,106
464743,2004739,"Donnelly, Ruth",USA,F,nt,nt,nt,2030,<NA>,<NA>,1920,wi,False,106
727004,611514,"Huguet, Marcel",FRA,M,nt,nt,nt,1550,<NA>,<NA>,1920,i,False,106
858601,24114065,"Kinin, Igor",RUS,M,nt,nt,nt,2013,1995,1948,1920,i,False,106
870133,24635049,"Koch, Horst",GER,M,nt,nt,nt,1847,<NA>,<NA>,1920,i,False,106
896750,24114081,"Kramorev, Yuri",RUS,M,nt,nt,nt,1975,<NA>,<NA>,1920,i,False,106
921631,4134028,"Kuznetsov, Fiodor",RUS,M,nt,nt,nt,1947,<NA>,<NA>,1920,i,False,106
949438,22718273,"Lederman, Isaac",BRA,M,nt,nt,nt,<NA>,1627,<NA>,1920,a,True,106
1178693,340219,"Muzik, Ferdinand",CZE,M,nt,nt,nt,1743,<NA>,<NA>,1920,i,False,106


### 9.3. Creación de Categorías por Edad

Clasificación en categorías oficiales de ajedrez juvenil:

| Categoría | Rango de Edad |
| :--- | :--- |
| Sub 8 | 0 – 7 años |
| Sub 10 | 8 – 9 años |
| Sub 12 | 10 – 11 años |
| Sub 14 | 12 – 13 años |
| Sub 16 | 14 – 15 años |
| Sub 18 | 16 – 17 años |
| Sub 20 | 18 – 19 años |
| Absoluta | 20+ años |

In [56]:
orden_jerarquico = [
    "Sub 8", "Sub 10", "Sub 12", "Sub 14", 
    "Sub 16", "Sub 18", "Sub 20", "Absoluta"
]

reglas_ajedrez = {
    "Sub 8":  (df_limpio["edad"] > 0) & (df_limpio["edad"] < 8),
    "Sub 10": (df_limpio["edad"] >= 8) & (df_limpio["edad"] < 10),
    "Sub 12": (df_limpio["edad"] >= 10) & (df_limpio["edad"] < 12),
    "Sub 14": (df_limpio["edad"] >= 12) & (df_limpio["edad"] < 14),
    "Sub 16": (df_limpio["edad"] >= 14) & (df_limpio["edad"] < 16),
    "Sub 18": (df_limpio["edad"] >= 16) & (df_limpio["edad"] < 18),
    "Sub 20": (df_limpio["edad"] >= 18) & (df_limpio["edad"] < 20),
    "Absoluta": df_limpio["edad"] >= 20,
}

df_limpio["categoria"] = np.select(
    list(reglas_ajedrez.values()), 
    list(reglas_ajedrez.keys()), 
    default="Sin Categoría"
)


df_limpio['categoria'] = pd.Categorical(
    df_limpio['categoria'], 
    categories=orden_jerarquico, 
    ordered=True
)



## 10. Segmentación de Jugadores Activos

Creación del subconjunto `df_activos`:
- Excluye flags `'i'` (inactivo) y `'wi'` (mujer inactiva).
- Trunca edad máxima a 95 años (eliminar outliers inverosímiles).

In [57]:

df_activos = df_limpio[~df_limpio['flag'].isin(['i', 'wi'])].copy()
df_activos = df_activos[df_activos['edad'] <= 95].copy()

df_activos.describe()

,rating,rapid_rating,blitz_rating,birthday,edad
count,219417.0,324549.0,203133.0,437166.0,437166.0
mean,1734.254137,1657.180201,1703.732402,1994.075706,31.924294
std,223.856323,200.865953,216.166838,19.752018,19.752018
min,1400.0,1400.0,1400.0,1931.0,4.0
25%,1556.0,1501.0,1533.0,1980.0,16.0
50%,1700.0,1611.0,1664.0,2001.0,25.0
75%,1873.0,1768.0,1833.0,2010.0,46.0
max,2823.0,2803.0,2860.0,2022.0,95.0


In [58]:
df_activos.info()

<class 'pandas.DataFrame'>
Index: 437166 entries, 6 to 1915093
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   fideid        437166 non-null  string  
 1   name          437166 non-null  string  
 2   country       437166 non-null  category
 3   sex           437166 non-null  category
 4   title         437166 non-null  category
 5   w_title       437166 non-null  category
 6   o_title       437166 non-null  category
 7   rating        219417 non-null  Int64   
 8   rapid_rating  324549 non-null  Int64   
 9   blitz_rating  203133 non-null  Int64   
 10  birthday      437166 non-null  Int64   
 11  flag          437166 non-null  category
 12  es_activo     437166 non-null  bool    
 13  edad          437166 non-null  Int64   
 14  categoria     437166 non-null  category
dtypes: Int64(5), bool(1), category(7), string(2)
memory usage: 43.6 MB


## 11. Exportación de Datasets Procesados

Se generan dos archivos finales en `data/processed/`:
- **`fide_players_all.parquet`** — Todos los jugadores con rating > 0 y birthday válido.
- **`fide_players_active.parquet`** — Solo jugadores activos con edad ≤ 95.

In [59]:
df_limpio.to_parquet('../data/processed/fide_players_all.parquet', index=False)

df_activos.to_parquet('../data/processed/fide_players_active.parquet', index=False)

print(f"   - Todos los jugadores: {len(df_limpio)} registros guardados en 'fide_players_all.parquet'")
print(f"   - Jugadores activos: {len(df_activos)} registros guardados en 'fide_players_active.parquet'")

   - Todos los jugadores: 776188 registros guardados en 'fide_players_all.parquet'
   - Jugadores activos: 437166 registros guardados en 'fide_players_active.parquet'
